# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

md = dataset.metadata

print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets by their `@id`, with associated field (column) ids and labels

def display_record_sets(ds):
    print("\nAvailable Record Sets:")
    for rs in ds.metadata.record_sets:
        print(f"\nRecord set: @id='{rs.id}' name='{rs.name}'")
        print("  Fields / Columns:")
        fields = rs.fields
        for field in fields:
            print(f"    @id='{field.id}'  label='{field.label}'  dataType='{field.data_type}'")

display_record_sets(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.


In [ ]:
# Identify available record set `@id`s for extraction
record_sets = [rs.id for rs in dataset.metadata.record_sets]

# Load all record sets into separate dataframes using their @id
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# For demonstration, pick the first record set for overview (replace with actual @id as needed)
example_record_set_id = record_sets[0] if record_sets else None
if example_record_set_id:
    print(f"Columns in record set {example_record_set_id}:\n", dataframes[example_record_set_id].columns.tolist())
    print(dataframes[example_record_set_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Adapt field IDs from previous cell output -- use actual field `@id` for robust/consistent code
record_set_id = example_record_set_id
df = dataframes[record_set_id]

# Try to automatically select a numeric field (float or int), or set manually:
numeric_field = None
for field in dataset.metadata.get_record_set(record_set_id).fields:
    if field.data_type in ('Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number') and field.id in df.columns:
        numeric_field = field.id
        break

if numeric_field is None:
    print("No numeric field found to process. Please set one manually if needed.")
else:
    # Drop missing values in the numeric field
    filtered_df = df[df[numeric_field].notnull()]

    # Filter: keep records where value > threshold (pick threshold by field typical values)
    threshold = filtered_df[numeric_field].mean() if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]) else 0
    try:
        # Ensure comparison works for numbers only
        filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
    except Exception:
        print(f"Cannot apply numeric filter to field '{numeric_field}', skipping filter.")

    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by another field if a suitable categorical field exists (string type)
    group_field = None
    for field in dataset.metadata.get_record_set(record_set_id).fields:
        # Exclude numeric field, and look for text/categorical fields with low cardinality
        if field.id != numeric_field and field.data_type in ('Text', 'schema:Text', 'String') and field.id in df.columns:
            unique_vals = df[field.id].nunique()
            if unique_vals > 1 and unique_vals < 20:
                group_field = field.id
                break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())
    else:
        print("\nNo suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple visualizations only if we have numeric fields
if example_record_set_id and numeric_field and not df.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()
    
    # If grouping field found, visualize grouped mean
    if 'group_field' in locals() and group_field:
        grouped_means = df.groupby(group_field)[numeric_field].mean()
        grouped_means = grouped_means.sort_values(ascending=False).head(10)
        plt.figure(figsize=(10, 4))
        grouped_means.plot(kind='bar')
        plt.ylabel(f"Mean {numeric_field}")
        plt.title(f"Mean of {numeric_field} by {group_field} (Top 10)")
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook provided a step-by-step guide using [`mlcroissant`](https://github.com/mlcommons/croissant/python) to explore the dataset *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells*.

- We loaded structured metadata, examined available record sets and fields using their `@id`, and loaded actual tabular data.
- We identified and analyzed numeric fields, demonstrated filtering, normalization, and, when feasible, grouping and visualizations.

For reproducible, standards-based data analysis: always reference dataset elements by `@id` as in this notebook for unambiguous processing. For detailed field or schema exploration, consult the [Croissant](https://mlcommons.org/croissant/) documentation or the dataset's JSON-LD schema directly.